In [1]:
?approx

approxfun {stats},R Documentation
"x, y",numeric vectors giving the coordinates of the points to be interpolated. Alternatively a single plotting structure can be specified: see xy.coords.
xout,an optional set of numeric values specifying where interpolation is to take place.
method,"specifies the interpolation method to be used. Choices are ""linear"" or ""constant""."
n,"If xout is not specified, interpolation takes place at n equally spaced points spanning the interval [min(x), max(x)]."
yleft,the value to be returned when input x values are less than min(x). The default is defined by the value of rule given below.
yright,the value to be returned when input x values are greater than max(x). The default is defined by the value of rule given below.
rule,"an integer (of length 1 or 2) describing how interpolation is to take place outside the interval [min(x), max(x)]. If rule is 1 then NAs are returned for such points and if it is 2, the value at the closest data extreme is used. Use, e.g., rule = 2:1, if the left and right side extrapolation should differ."
f,"for method = ""constant"" a number between 0 and 1 inclusive, indicating a compromise between left- and right-continuous step functions. If y0 and y1 are the values to the left and right of the point then the value is y0 if f == 0, y1 if f == 1, and y0*(1-f)+y1*f for intermediate values. In this way the result is right-continuous for f == 0 and left-continuous for f == 1, even for non-finite y values."
ties,"handling of tied x values. The string ""ordered"" or a function (or the name of a function) taking a single vector argument and returning a single number or a list of both, e.g., list(""ordered"", mean), see ‘Details’."
na.rm,"logical specifying how missing values (NA's) should be handled. Setting na.rm=FALSE will propagate NA's in y to the interpolated values, also depending on the rule set. Note that in this case, NA's in x are invalid, see also the examples."


In [2]:
?spline

splinefun {stats},R Documentation
"x, y","vectors giving the coordinates of the points to be interpolated. Alternatively a single plotting structure can be specified: see xy.coords. y must be increasing or decreasing for method = ""hyman""."
m,"(for splinefunH()): vector of slopes m_i at the points (x_i,y_i); these together determine the Hermite “spline” which is piecewise cubic, (only) once differentiable continuously."
method,"specifies the type of spline to be used. Possible values are ""fmm"", ""natural"", ""periodic"", ""monoH.FC"" and ""hyman"". Can be abbreviated."
n,"if xout is left unspecified, interpolation takes place at n equally spaced points spanning the interval [xmin, xmax]."
"xmin, xmax",left-hand and right-hand endpoint of the interpolation interval (when xout is unspecified).
xout,an optional set of values specifying where interpolation is to take place.
ties,"handling of tied x values. The string ""ordered"" or a function (or the name of a function) taking a single vector argument and returning a single number or a length-2 list of both, see approx and its ‘Details’ section, and the example below."


### Check how splines work and how to adjust them:

In [3]:
fitSplines(lower=lower,upper=upper,vals=vals[,1],probs=probs[,1],cn,degree=bs_degree,width=width,spline="MP",exponential_tails=exponential_tails,rtp=rtp,ltp=ltp,edr=edr)

ERROR: Error in fitSplines(lower = lower, upper = upper, vals = vals[, 1], probs = probs[, : could not find function "fitSplines"


In [ ]:




fitSplines <- function(lower, upper, vals, probs,degree,cn,width,spline="NS",exponential_tails=exponential_tails,ltp=ltp,rtp=rtp,edr=edr){
    print(paste("Cn:",cn))
    extra_points=NULL
# Adjust the cumulative probabilities to account for the limits
    extended_vals <- c(lower, vals, upper)
    extended_cumsums <- c(0, probs, 1)

    print("b")
    print(paste("x<-c(",paste(unlist(extended_vals),collapse=","),")",sep=""))
    print(paste("y<-c(",paste(unlist(extended_cumsums),collapse=","),")",sep=""))


    if(spline=="MP"){
        mid_points <- vals-(width/2)
        extended_vals <- c(lower, as.vector(rbind(mid_points, vals)),max(vals)+(width/2), upper)
        print(paste("x<-c(",paste(unlist(extended_vals),collapse=","),")",sep=""))

        mid_cumsums <- c(probs[1]/2, (probs[-1] - (diff(probs)/2)))
        print(mid_cumsums)
        print(probs)
        
        extended_cumsums <- c(0, as.vector(rbind(mid_cumsums,probs)),(probs[length(probs)]+((1-probs[length(probs)])/2)), 1)
         print(paste("y<-c(",paste(unlist(extended_cumsums),collapse=","),")",sep=""))

  
       #extended_cumsums <- c(0, as.vector(rbind(probs-(probs[1]/2),probs)),probs[length(probs)-1]+(probs[1]/2), 0)
       #   print(paste("y<-c(",paste(unlist(extended_cumsums),collapse=","),")",sep=""))


        # The fisrt one works because it is a cumulative sum actually
        
    }

    if(exponential_tails==TRUE){
    # 🔹 Add extra points (exponential growth) - BETWEEN 1rs and second values
      print(ltp)
       print(rtp)
       print(edr)
        extra_left_vals <- seq(extended_vals[1], extended_vals[2], length.out = ltp)[-1]  # Avoid duplicate 0
        print(extra_left_vals)
        extra_left_cumsums <- extended_cumsums[2] * exp(edr * (extra_left_vals - extended_vals[2]))  # Starts slow, then speeds up - 1 can be adjusted
        # 🔹 Add extra points **between 40 and 100** (exponential decay)
        extra_right_vals <- seq(extended_vals[length(extended_vals)-1], extended_vals[length(extended_vals)], length.out = rtp)[-1]  # Avoid duplicate values
        extra_right_cumsums <- extended_cumsums[length(extended_cumsums)-1] + (1 - extended_cumsums[length(extended_cumsums)-1]) * (1 - exp(-edr *(extra_right_vals - extended_vals[length(extended_vals)-1])))  # Approaching 1

    # Set aside a data frame for plotting
    extra_points<-data.frame(x=c(extra_left_vals,extra_right_vals),y=c(extra_left_cumsums,extra_right_cumsums))
    print(extra_points)

    extra_points<-extra_points[!(extra_points$x%in%extended_vals),]
    print(extra_points)
    # Get rid of unnecessary data
    extra_left_cumsums <- extra_left_cumsums[-(length(extra_left_cumsums))]
    extra_right_cumsums <- extra_right_cumsums[-length(extra_right_cumsums)] # Avoid duplicate probability at the end
    
    # 🔹 Combine all data
    new_vals <- c(0, extra_left_vals[-length(extra_left_vals)], extended_vals[c(-1,-length(extended_vals))], extra_right_vals)
    new_cumsums <- c(0, extra_left_cumsums, extended_cumsums[c(-1,-length(extended_cumsums))], extra_right_cumsums,1)

    extended_vals<-new_vals
    extended_cumsums<-new_cumsums

    print(paste("x<-c(",paste(unlist(extended_vals),collapse=","),")",sep=""))
    print(paste("y<-c(",paste(unlist(extended_cumsums),collapse=","),")",sep=""))
    
    }

    #cdf_bspline_fit <- lm(extended_cumsums ~ bSpline(extended_vals, degree = degree))
    if(spline=="NS"){
    #cdf_bspline_fit <- lm(extended_cumsums ~ ns(extended_vals, df= length(extended_cumsums)-3)) # Fit the natural cubic spline
    cdf_bspline_fit <- lm(extended_cumsums ~ ns(extended_vals, df=degree))
    }else if(spline=="MP"){
        
    cdf_bspline_fit <- scam(extended_cumsums ~ s(extended_vals, bs = "mpi"))
     
    
    } 
    
    new_data<-seq(lower,upper,length.out=100) # Fit it to new data
    pdf_values <- predict(cdf_bspline_fit,newdata = data.frame(extended_vals=new_data))

    print(pdf_values)

    cdf_bspline_derivative <- c(diff(pdf_values) / diff(new_data),0)

   extended_pdf_vals <- cdf_bspline_derivative 

    # Compute the integral of the PDF (sum of areas)
    pdf_integral <- sum(extended_pdf_vals) * (upper - lower) / length(extended_pdf_vals)

    # Normalize the PDF (make sure the integral is 1)
    pdf_normalized <- extended_pdf_vals / pdf_integral


    #print(paste("pdf_normalized:",length(pdf_normalized)))
    b_spline <- list(fit.cdf=cdf_bspline_fit,cdf=pdf_values,pdf_normalized=pdf_normalized,values=new_data,extra_points=extra_points)
    #b_spline <- list(fit.cdf=pdf_bspline_fit,pdf_normalized=smooth_pdf_values,values=smooth_x)
        return(b_spline)
    
}

In [4]:
?scam

No documentation for ‘scam’ in specified packages and libraries:
you could try ‘??scam’

In [5]:
library(splines2)
library(splines)
library(scam)

Loading required package: mgcv

Loading required package: nlme

This is mgcv 1.9-1. For overview type 'help("mgcv-package")'.

This is scam 1.2-18.



In [7]:
?s

s {mgcv},R Documentation
...,"a list of variables that are the covariates that this smooth is a function of. Transformations whose form depends on the values of the data are best avoided here: e.g. s(log(x)) is fine, but s(I(x/sd(x))) is not (see predict.gam)."
k,"the dimension of the basis used to represent the smooth term. The default depends on the number of variables that the smooth is a function of. k should not be less than the dimension of the null space of the penalty for the term (see null.space.dimension), but will be reset if it is. See choose.k for further information."
fx,indicates whether the term is a fixed d.f. regression spline (TRUE) or a penalized regression spline (FALSE).
bs,"a two letter character string indicating the (penalized) smoothing basis to use. (eg ""tp"" for thin plate regression spline, ""cr"" for cubic regression spline). see smooth.terms for an over view of what is available."
m,"The order of the penalty for this term (e.g. 2 for normal cubic spline penalty with 2nd derivatives when using default t.p.r.s basis). NA signals autoinitialization. Only some smooth classes use this. The ""ps"" class can use a 2 item array giving the basis and penalty order separately."
by,"a numeric or factor variable of the same dimension as each covariate. In the numeric vector case the elements multiply the smooth, evaluated at the corresponding covariate values (a ‘varying coefficient model’ results). For the numeric by variable case the resulting smooth is not usually subject to a centering constraint (so the by variable should not be added as an additional main effect). In the factor by variable case a replicate of the smooth is produced for each factor level (these smooths will be centered, so the factor usually needs to be added as a main effect as well). See gam.models for further details. A by variable may also be a matrix if covariates are matrices: in this case implements linear functional of a smooth (see gam.models and linear.functional.terms for details)."
xt,"Any extra information required to set up a particular basis. Used e.g. to set large data set handling behaviour for ""tp"" basis. If xt$sumConv exists and is FALSE then the summation convention for matrix arguments is turned off."
id,"A label or integer identifying this term in order to link its smoothing parameters to others of the same type. If two or more terms have the same id then they will have the same smoothing paramsters, and, by default, the same bases (first occurance defines basis type, but data from all terms used in basis construction). An id with a factor by variable causes the smooths at each factor level to have the same smoothing parameter."
sp,any supplied smoothing parameters for this term. Must be an array of the same length as the number of penalties for this smooth. Positive or zero elements are taken as fixed smoothing parameters. Negative elements signal auto-initialization. Over-rides values supplied in sp argument to gam. Ignored by gamm.
pc,"If not NULL, signals a point constraint: the smooth should pass through zero at the point given here (as a vector or list with names corresponding to the smooth names). Never ignored if supplied. See identifiability."


In [8]:
?smooth.terms

smooth.terms {mgcv},R Documentation
